# Shortlister training (Role A)

QLoRA fine-tuning of the misconception shortlister as a **bi-encoder**: the query from
`build_query()` and every Eedi misconception name are embedded by the same model, and
misconceptions are ranked by cosine similarity. Misconceptions never seen in training can
still be ranked, because they are scored by their text.

**Kaggle setup**
1. Accelerator: **GPU T4 x1** (or P100). Internet: **on**.
2. *Add Input* -> the competition **eedi-mining-misconceptions-in-mathematics**.
3. Our code, either way:
   - *Add Input* -> a private dataset named **matheasy-code** holding the repo (at least `matheasy/`), or
   - *Add-ons -> Secrets* -> `GITHUB_TOKEN` with read access to Zenith053/MathEasy.

`SMOKE = True` is a quick check: 100 training pairs, 1 epoch.
For the real runs set `SMOKE = False` and pick `SIZE`: run 1 = `"3B"`, run 2 = `"1.5B"`.
Run 3 = `USE_LLM = True`: adds the LLM-written examples (`data/eedi/llm_examples.csv`, made by
`python -m matheasy.shortlister.llm_examples`) to training. Attach that file as a private
dataset (*Add Input*); it is found wherever it is mounted.
Run 4 = `USE_RULES = True` as well: adds the rule-generated pairs (made here by
`python -m matheasy.shortlister.rule_pairs` from B's bank, so nothing needs attaching).
The adapter from the epoch with the best val MAP@25 is kept; test is reported for that epoch only.

In [ ]:
!pip install -q -U peft bitsandbytes lark rank_bm25

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/tmp/MathEasy")  # not /kaggle/working: Eedi-derived files must not end up in the saved output
if not WORK.exists():
    code_dir = next(INPUT.rglob("prepare_eedi.py"), None)  # matheasy-code dataset, wherever it is mounted
    if code_dir:
        # The package folder is the one above shortlister/; Kaggle may or may not keep "matheasy/" in the path.
        shutil.copytree(code_dir.parents[1], WORK / "matheasy")
        # B's rules read eval/held_back_rules.json, and rule_pairs.py uses eval/build_probes.py.
        shutil.copytree(next(INPUT.rglob("held_back_rules.json")).parent, WORK / "eval")
    else:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
        subprocess.run(["git", "clone", "--depth", "1",
                        f"https://{token}@github.com/Zenith053/MathEasy.git", str(WORK)], check=True)

# The scripts read Eedi from data/eedi/raw: point it at the competition input.
eedi = next(INPUT.rglob("misconception_mapping.csv")).parent
raw = WORK / "data" / "eedi" / "raw"
raw.parent.mkdir(parents=True, exist_ok=True)
if not raw.exists():
    raw.symlink_to(eedi)
print("code:", WORK, " eedi:", eedi)

os.chdir(WORK)
sys.path.insert(0, str(WORK))
!python -m matheasy.shortlister.prepare_eedi
!python -m matheasy.shortlister.query
!python -m matheasy.shortlister.hard_negatives

In [ ]:
SMOKE = True
SIZE = "3B"                          # run 1: "3B", run 2: "1.5B"
USE_LLM = False                      # run 3: add LLM-written examples for misconceptions with no training question
USE_RULES = False                    # run 4: add rule-generated pairs (live-tutor query format)

MODEL = f"Qwen/Qwen2.5-{SIZE}-Instruct"
N_TRAIN = 100 if SMOKE else None     # None = all training pairs
EPOCHS = 1 if SMOKE else 3
BATCH = 8                            # in-batch negatives: bigger is better if memory allows
LR = 1e-4
TEMP = 0.05
N_HARD = 3                           # hard negatives per pair (0 = in-batch negatives only)
MAX_Q, MAX_M = 512, 96               # token limits for query and misconception name (longest: ~870 / ~210 chars)
LORA_R = 16
SEED = 20261016
WARMUP = 0.1                         # share of steps with rising learning rate
OUT = Path(f"/kaggle/working/adapter-{SIZE}" + ("-llm" if USE_LLM else "") + ("-rules" if USE_RULES else "")
           + ("-smoke" if SMOKE else ""))

INSTRUCT = "Instruct: Given a student's wrong step, retrieve the misconception that explains it\nQuery: "

In [ ]:
import pandas as pd
from matheasy.shortlister.shortlist import map_at_k, recall_at_k

queries = pd.read_csv("data/eedi/queries.csv")
names = pd.read_csv("data/eedi/raw/misconception_mapping.csv")
names["MisconceptionName"] = names.MisconceptionName.str.strip()

train = queries[queries.Split == "train"].sample(frac=1, random_state=SEED)
if N_TRAIN:
    train = train.head(N_TRAIN)
val = queries[queries.Split == "val"]
test = queries[queries.Split == "test"]
seen = set(queries[queries.Split == "train"].MisconceptionId)  # misconceptions with training questions
print(len(train), "train pairs,", len(val), "val pairs,", len(test), "test pairs,",
      len(names), "candidate misconceptions")
name_of = dict(zip(names.MisconceptionId, names.MisconceptionName))

# Hard negatives: siblings (other wrong options of the same question) and look-alike names.
negs = pd.read_csv("data/eedi/train_negatives.csv")
negs_of = {(r.QuestionId, r.WrongAnswer): [int(m) for m in r.NegativeIds.split()][:N_HARD]
           for r in negs.itertuples()}

from matheasy.shortlister.hard_negatives import mine


def add_extra(train, extra, tag):
    # Each extra row is its own pseudo-question (no siblings), so its negatives are look-alike names.
    # Keys are (pseudo-question, tag), so LLM and rule rows never clash.
    extra = extra.reset_index(drop=True)
    extra = extra.assign(QuestionId=-1 - extra.index, WrongAnswer=tag)
    for r in mine(extra, names).itertuples():
        negs_of[(r.QuestionId, r.WrongAnswer)] = [int(m) for m in r.NegativeIds.split()][:N_HARD]
    print(f"+ {len(extra)} {tag} pairs")
    return pd.concat([train, extra[["QuestionId", "WrongAnswer", "MisconceptionId", "Query"]]])


if USE_LLM:
    llm = pd.read_csv(next(INPUT.rglob("llm_examples.csv")))
    train = add_extra(train, llm[llm.Topic != "other"].dropna(subset=["Query"]), "llm")
if USE_RULES:
    !python -m matheasy.shortlister.rule_pairs
    rules = pd.read_csv("data/eedi/rule_pairs.csv")
    train = add_extra(train, rules, "rule")
train = train.sample(frac=1, random_state=SEED)
print(len(train), "training pairs")

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

torch.manual_seed(SEED)
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "right"
model = AutoModel.from_pretrained(
    MODEL,
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                           bnb_4bit_compute_dtype=torch.float16,
                                           bnb_4bit_use_double_quant=True),
    dtype=torch.float16, device_map={"": 0})
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True,
                                        gradient_checkpointing_kwargs={"use_reentrant": False})
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()


def embed(texts, max_len):
    """Last-token pooling, L2-normalised."""
    batch = tok([t + tok.eos_token for t in texts], max_length=max_len, truncation=True,
                padding=True, return_tensors="pt").to(model.device)
    hidden = model(**batch).last_hidden_state
    last = batch["attention_mask"].sum(1) - 1
    return F.normalize(hidden[torch.arange(len(texts)), last].float(), dim=-1)


@torch.no_grad()
def evaluate(df, k=25):
    model.eval()
    m = torch.cat([embed(names.MisconceptionName[i:i + 64].tolist(), MAX_M)
                   for i in range(0, len(names), 64)])
    q = torch.cat([embed([INSTRUCT + t for t in df.Query[i:i + 16]], MAX_Q)
                   for i in range(0, len(df), 16)])
    top = (q @ m.T).topk(k, dim=1).indices.cpu()
    ranked = [[names.MisconceptionId.iloc[j] for j in row] for row in top.tolist()]
    true = df.MisconceptionId.tolist()
    model.train()
    unseen = [t not in seen for t in true]
    pick = lambda keep: ([r for r, u in zip(ranked, unseen) if u == keep], [t for t, u in zip(true, unseen) if u == keep])
    return {"MAP@25": map_at_k(ranked, true, k), "Recall@10": recall_at_k(ranked, true, 10),
            "MAP@25 seen": map_at_k(*pick(False), k), "MAP@25 unseen": map_at_k(*pick(True), k)}


show = lambda r: "  ".join(f"{name} {v:.3f}" for name, v in r.items())

In [ ]:
import time
from transformers import get_linear_schedule_with_warmup

print("before training, val: ", show(evaluate(val)))

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
steps = EPOCHS * (len(train) // BATCH)
sched = get_linear_schedule_with_warmup(opt, int(WARMUP * steps), steps)
scaler = torch.amp.GradScaler()
best = None
model.train()
start = time.time()
for epoch in range(EPOCHS):
    shuffled = train.sample(frac=1, random_state=SEED + epoch)
    for i in range(0, len(shuffled) - BATCH + 1, BATCH):
        b = shuffled.iloc[i:i + BATCH]
        # Candidates: the batch's true misconceptions, then each pair's hard negatives.
        cand = b.MisconceptionId.tolist() + [m for key in zip(b.QuestionId, b.WrongAnswer)
                                             for m in negs_of.get(key, [])]
        with torch.autocast("cuda", dtype=torch.float16):
            q = embed([INSTRUCT + t for t in b.Query], MAX_Q)
            p = embed([name_of[m] for m in cand], MAX_M)
        logits = (q @ p.T) / TEMP
        # A candidate with the row's own misconception is not a negative (except its own column).
        pos = torch.tensor(b.MisconceptionId.values, device=logits.device)
        same = pos[:, None] == torch.tensor(cand, device=logits.device)[None, :]
        same[torch.arange(len(b)), torch.arange(len(b))] = False
        loss = F.cross_entropy(logits.masked_fill(same, -1e4), torch.arange(len(b), device=logits.device))
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        if (i // BATCH) % 5 == 0:
            print(f"epoch {epoch} step {i // BATCH:4d}  loss {loss.item():.3f}  {time.time() - start:.0f}s")
    v = evaluate(val)
    print(f"epoch {epoch}, val: ", show(v))
    if best is None or v["MAP@25"] > best["val"]["MAP@25"]:
        # Keep this epoch's adapter; test is only read for the epoch chosen on val.
        best = {"epoch": epoch, "val": v, "test": evaluate(test)}
        model.save_pretrained(OUT)
        tok.save_pretrained(OUT)

In [ ]:
import json
print(f"best epoch {best['epoch']} ({SIZE}, N_HARD={N_HARD}, USE_LLM={USE_LLM}, USE_RULES={USE_RULES})")
print("  val: ", show(best["val"]))
print("  test:", show(best["test"]))
print("BM25 placeholder (bare-topic queries): val MAP@25 0.047 Recall@10 0.069, test MAP@25 0.038 Recall@10 0.059")
# Scores only (no Eedi text), so they are safe to keep in the output.
(OUT / "results.json").write_text(json.dumps({"model": MODEL, "n_hard": N_HARD, "epochs": EPOCHS, "use_llm": USE_LLM, "use_rules": USE_RULES,
                                              "smoke": SMOKE, **best}, indent=1))
print("saved adapter ->", OUT, sorted(p.name for p in OUT.iterdir()))

**Smoke test passes when:** the loss goes down, the evaluations print, and the adapter folder
with `results.json` is saved. With only 100 pairs the scores are not meaningful; the real runs are.